<a href="https://colab.research.google.com/github/PiyushGit11/Generative-AI-internship/blob/main/Day8/Presentation/MultilingualNews_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Multilingual News Translator & Summarizer v2
### Powered by NLLB-200 (Meta) + mT5-large

| Component | Model | Why |
|---|---|---|
| **Translation** | `facebook/nllb-200-distilled-600M` | Supports 200 languages, far better than Helsinki |
| **Summarization** | `csebuetnlp/mT5_multilingual_XLSum` | Trained on 45 languages of news data |

> ⚡ **Recommended:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Cell 1 — Install (run once, then restart runtime)
!pip install -q transformers==4.44.2 sentencepiece langdetect accelerate torch
print("✅ Done — restart runtime if this is your first run")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.3 MB/s eta 0:00:00
✅ Done — restart runtime if this is your first run


In [ ]:
# Cell 2 — Imports & device setup
import torch
from langdetect import detect, DetectorFactory
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

# Make language detection deterministic
DetectorFactory.seed = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Imports OK | Device: {'GPU 🚀' if DEVICE == 'cuda' else 'CPU (slower)'}")

✅ Imports OK | Device: GPU 🚀


In [ ]:
# Cell 3 — NLLB language code mapping
# NLLB uses BCP-47 style codes like 'mal_Mlym' instead of 'ml'
# Full list: https://github.com/facebookresearch/flores/blob/main/flores200/README.md

LANGDETECT_TO_NLLB = {
    "af": "afr_Latn",  "ar": "arb_Arab",  "az": "azj_Latn",
    "be": "bel_Cyrl",  "bg": "bul_Cyrl",  "bn": "ben_Beng",
    "ca": "cat_Latn",  "cs": "ces_Latn",  "cy": "cym_Latn",
    "da": "dan_Latn",  "de": "deu_Latn",  "el": "ell_Grek",
    "en": "eng_Latn",  "es": "spa_Latn",  "et": "est_Latn",
    "fa": "pes_Arab",  "fi": "fin_Latn",  "fr": "fra_Latn",
    "ga": "gle_Latn",  "gl": "glg_Latn",  "gu": "guj_Gujr",
    "he": "heb_Hebr",  "hi": "hin_Deva",  "hr": "hrv_Latn",
    "hu": "hun_Latn",  "hy": "hye_Armn",  "id": "ind_Latn",
    "it": "ita_Latn",  "ja": "jpn_Jpan",  "ka": "kat_Geor",
    "kn": "kan_Knda",  "ko": "kor_Hang",  "lt": "lit_Latn",
    "lv": "lvs_Latn",  "mk": "mkd_Cyrl",  "ml": "mal_Mlym",  # Malayalam
    "mr": "mar_Deva",  "ms": "zsm_Latn",  "mt": "mlt_Latn",
    "nl": "nld_Latn",  "no": "nob_Latn",  "pa": "pan_Guru",
    "pl": "pol_Latn",  "pt": "por_Latn",  "ro": "ron_Latn",
    "ru": "rus_Cyrl",  "sk": "slk_Latn",  "sl": "slv_Latn",
    "sq": "als_Latn",  "sr": "srp_Cyrl",  "sv": "swe_Latn",
    "sw": "swh_Latn",  "ta": "tam_Taml",  "te": "tel_Telu",
    "th": "tha_Thai",  "tl": "tgl_Latn",  "tr": "tur_Latn",
    "uk": "ukr_Cyrl",  "ur": "urd_Arab",  "vi": "vie_Latn",
    "zh-cn": "zho_Hans", "zh-tw": "zho_Hant", "zh": "zho_Hans",
}

def detect_language(text):
    try:
        lang = detect(text)
        nllb_code = LANGDETECT_TO_NLLB.get(lang, None)
        print(f"[Detected]: langdetect='{lang}' → NLLB='{nllb_code}'")
        return lang, nllb_code
    except Exception as e:
        print(f"[Detection error]: {e} → defaulting to English")
        return "en", "eng_Latn"

print("✅ Language mapping ready (200 languages)")

✅ Language mapping ready (200 languages)


In [ ]:
# Cell 4 — Load NLLB-200 Translation Model (~1.2GB, loads once)
# facebook/nllb-200-distilled-600M — best accuracy/size tradeoff
# Supports 200 languages vs Helsinki's ~60

NLLB_MODEL = "facebook/nllb-200-distilled-600M"
print(f"[Loading translation model: {NLLB_MODEL}]")
print("(~1.2GB download, takes 1-2 mins on first run)")

nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)

if DEVICE == "cuda":
    nllb_model = nllb_model.to("cuda")

print("✅ NLLB-200 translation model loaded")

def translate_to_english(text, src_nllb_code):
    if src_nllb_code == "eng_Latn" or src_nllb_code is None and "en" in text[:20]:
        print("[Translation]: Already English, skipping.")
        return text

    if src_nllb_code is None:
        print("[Warning]: Unknown language code, attempting translation anyway.")
        src_nllb_code = "eng_Latn"

    # Tokenize with forced source language
    nllb_tokenizer.src_lang = src_nllb_code
    target_lang_id = nllb_tokenizer.convert_tokens_to_ids("eng_Latn")

    # Chunk long texts to fit within model limits
    words = text.split()
    chunk_size = 200  # conservative for NLLB
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    print(f"[Translating {len(chunks)} chunk(s)...]")

    translated_chunks = []
    for i, chunk in enumerate(chunks):
        inputs = nllb_tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        if DEVICE == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        translated_tokens = nllb_model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_new_tokens=512,
            num_beams=4
        )
        result = nllb_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
        translated_chunks.append(result[0])
        print(f"  Chunk {i+1}/{len(chunks)} done")

    return " ".join(translated_chunks)

[Loading translation model: facebook/nllb-200-distilled-600M]
(~1.2GB download, takes 1-2 mins on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB-200 translation model loaded


In [ ]:
# Cell 5 — Load mT5 Summarization Model (~1.1GB)
# Trained specifically on multilingual news from 45 languages (BBC XL-Sum dataset)
# Much better at news summarization than flan-t5-base

SUMMARIZER_MODEL = "csebuetnlp/mT5_multilingual_XLSum"
print(f"[Loading summarizer: {SUMMARIZER_MODEL}]")
print("(~1.1GB download, takes 1-2 mins on first run)")

# Special whitespace token required by this model
WHITESPACE_HANDLER = lambda k: k.strip().replace("\n", " ").replace("  ", " ")

mt5_tokenizer = AutoTokenizer.from_pretrained(SUMMARIZER_MODEL, use_fast=False)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZER_MODEL)

if DEVICE == "cuda":
    mt5_model = mt5_model.to("cuda")

print("✅ mT5 XLSum summarizer loaded")

def summarize_news(english_text, max_length=150, min_length=40):
    # Clean and truncate
    cleaned = WHITESPACE_HANDLER(english_text)
    words = cleaned.split()
    if len(words) > 600:
        cleaned = " ".join(words[:600])
        print("[Note]: Text truncated to 600 words.")

    inputs = mt5_tokenizer(
        cleaned,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )

    if DEVICE == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    output = mt5_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_length,
        min_length=min_length,
        num_beams=4,
        length_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return mt5_tokenizer.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)

[Loading summarizer: csebuetnlp/mT5_multilingual_XLSum]
(~1.1GB download, takes 1-2 mins on first run)


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

✅ mT5 XLSum summarizer loaded


In [ ]:
# Cell 6 — Full pipeline
def news_translate_and_summarize(news_text):
    print("\n" + "="*60)
    print("📰 INPUT (first 300 chars):")
    print(news_text[:300] + ("..." if len(news_text) > 300 else ""))
    print("="*60)

    # Step 1: Detect
    lang, nllb_code = detect_language(news_text)

    # Step 2: Translate
    print("\n🔄 Translating to English...")
    english_text = translate_to_english(news_text, nllb_code)
    print("\n[English translation (first 500 chars)]:")
    print(english_text[:500])

    # Step 3: Summarize
    print("\n✍️  Summarizing...")
    summary = summarize_news(english_text)

    print("\n" + "="*60)
    print("📋 FINAL SUMMARY (English):")
    print(summary)
    print("="*60)

    return {
        "detected_language": lang,
        "nllb_code": nllb_code,
        "english_translation": english_text,
        "summary": summary
    }

print("✅ Pipeline ready")

✅ Pipeline ready


In [ ]:
# Cell 8 — Test: Arabic news
arabic_news = """
أعلنت وكالة ناسا الأمريكية للفضاء اليوم عن اكتشاف علمي مهم يتعلق بالمريخ.
وأكد العلماء وجود آثار لمياه جوفية تحت سطح الكوكب الأحمر، مما يفتح آفاقاً جديدة
للبحث عن أشكال الحياة خارج كوكب الأرض. وأشارت الدراسة المنشورة في مجلة ساينس
إلى أن هذه المياه موجودة على عمق يتراوح بين 10 و20 كيلومتراً تحت سطح المريخ.
وقال مدير وكالة ناسا إن هذا الاكتشاف يمثل خطوة كبيرة نحو فهم إمكانية وجود حياة
في الفضاء الخارجي، مضيفاً أن البعثة المرتقبة إلى المريخ عام 2030 ستحمل معدات
متطورة للتحقق من هذه النتائج بشكل مباشر.
"""
result = news_translate_and_summarize(arabic_news)


📰 INPUT (first 300 chars):

أعلنت وكالة ناسا الأمريكية للفضاء اليوم عن اكتشاف علمي مهم يتعلق بالمريخ.
وأكد العلماء وجود آثار لمياه جوفية تحت سطح الكوكب الأحمر، مما يفتح آفاقاً جديدة
للبحث عن أشكال الحياة خارج كوكب الأرض. وأشارت الدراسة المنشورة في مجلة ساينس
إلى أن هذه المياه موجودة على عمق يتراوح بين 10 و20 كيلومتراً تحت سطح...
[Detected]: langdetect='ar' → NLLB='arb_Arab'

🔄 Translating to English...
[Translating 1 chunk(s)...]
  Chunk 1/1 done

[English translation (first 500 chars)]:
NASA announced today an important scientific discovery related to Mars. Scientists confirmed the presence of groundwater traces beneath the surface of the Red Planet, opening up new prospects for the search for life forms beyond Earth. The study, published in Science, indicated that this water exists at a depth of between 10 and 20 kilometers below the surface of Mars. The director of NASA said the discovery represents a major step toward understanding the possibility of life in outer space, add

✍️  

In [ ]:
# Cell 9 — Test: Hindi news
hindi_news = """
भारत सरकार ने आज एक नई शिक्षा नीति की घोषणा की है जिसके तहत देशभर के सरकारी
स्कूलों में डिजिटल शिक्षा को बढ़ावा दिया जाएगा। इस योजना के अंतर्गत 10 लाख
स्कूलों में टैबलेट और इंटरनेट की सुविधा प्रदान की जाएगी। शिक्षा मंत्री ने बताया
कि इस परियोजना पर 50,000 करोड़ रुपये खर्च किए जाएंगे और इससे 20 करोड़ छात्रों
को फायदा होगा। यह योजना 2025 तक पूरी तरह लागू की जाएगी और इसका मुख्य उद्देश्य
ग्रामीण क्षेत्रों के बच्चों को गुणवत्तापूर्ण शिक्षा उपलब्ध कराना है।
"""
result = news_translate_and_summarize(hindi_news)


📰 INPUT (first 300 chars):

भारत सरकार ने आज एक नई शिक्षा नीति की घोषणा की है जिसके तहत देशभर के सरकारी
स्कूलों में डिजिटल शिक्षा को बढ़ावा दिया जाएगा। इस योजना के अंतर्गत 10 लाख
स्कूलों में टैबलेट और इंटरनेट की सुविधा प्रदान की जाएगी। शिक्षा मंत्री ने बताया
कि इस परियोजना पर 50,000 करोड़ रुपये खर्च किए जाएंगे और इससे 20 करोड...
[Detected]: langdetect='hi' → NLLB='hin_Deva'

🔄 Translating to English...
[Translating 1 chunk(s)...]
  Chunk 1/1 done

[English translation (first 500 chars)]:
The Education Minister said that Rs 50,000 crore will be spent on the project and it will benefit 20 crore students. The scheme will be fully implemented by 2025 and its main objective is to provide quality education to children in rural areas.

✍️  Summarizing...

📋 FINAL SUMMARY (English):
The government has announced plans for a scheme to improve education for children in rural areas of Northern Ireland in the next five years. They will be based in the Isle of Wight.


In [ ]:
french_news = """On savait que le record finirait par tomber. Olivier Giroud l'aura tenu pendant à peine quatre ans et lui-même s'en était fait une raison. "Je lui souhaite de me dépasser dans quelques mois, pas trop tôt non plus, avait plaisanté l'attaquant du Losc dans une interview pour L'Equipe en octobre dernier(Nouvelle fenêtre). J'aimerais bien qu'il me passe devant pendant la Coupe du monde, ce serait sympa." C'est désormais chose faite."""
result = news_translate_and_summarize(french_news)


📰 INPUT (first 300 chars):
On savait que le record finirait par tomber. Olivier Giroud l'aura tenu pendant à peine quatre ans et lui-même s'en était fait une raison. "Je lui souhaite de me dépasser dans quelques mois, pas trop tôt non plus, avait plaisanté l'attaquant du Losc dans une interview pour L'Equipe en octobre dernie...
[Detected]: langdetect='fr' → NLLB='fra_Latn'

🔄 Translating to English...
[Translating 1 chunk(s)...]
  Chunk 1/1 done

[English translation (first 500 chars)]:
Olivier Giroud held the record for just four years and made a reason for himself. "I wish him to beat me in a few months, not too soon either", the Losc striker joked in an interview for L'Equipe last October.

✍️  Summarizing...

📋 FINAL SUMMARY (English):
French footballer Olivier Giroud has broken the world record for the best player in almost a decade, according to the French Football Federation (Fifa) .


In [ ]:
spanish_news = """ El texto del memorando de entendimiento del presidente Donald Trump con Irán, cuando finalmente se publique, puede que no disipe los temores de sus críticos ante un posible mal acuerdo.
Cada vez hay más indicios de que así será.
"""
result = news_translate_and_summarize(spanish_news)



📰 INPUT (first 300 chars):
 El texto del memorando de entendimiento del presidente Donald Trump con Irán, cuando finalmente se publique, puede que no disipe los temores de sus críticos ante un posible mal acuerdo.
Cada vez hay más indicios de que así será.

[Detected]: langdetect='es' → NLLB='spa_Latn'

🔄 Translating to English...
[Translating 1 chunk(s)...]
  Chunk 1/1 done

[English translation (first 500 chars)]:
The text of President Donald Trump's memorandum of understanding with Iran, when finally published, may not dispel his critics' fears of a possible bad deal.

✍️  Summarizing...

📋 FINAL SUMMARY (English):
US President Donald Trump has signed a deal with Iran to reach a post-nuclear deal with the country's nuclear weapons agency, which could lead to a diplomatic row.


In [ ]:



# Cell 10 — Paste your own article
print("Paste your news article (any language). Press Enter twice when done.")
lines = []
while True:
    line = input()
    if line == "":
        break
    lines.append(line)
news_input = "\n".join(lines)

if news_input.strip():
    result = news_translate_and_summarize(news_input)
else:
    print("No input provided.")

Paste your news article (any language). Press Enter twice when done.
